# Borrowing bound and the endogenous grid

This check preserves the buffer-stock income laws and the original savings grid at bound zero. It then verifies the boundary point computed from infinite marginal value. A **constant field** is an input whose value does not vary with the state; the bound `L` is one such input. This notebook checks one numerical step, not a stationary solution.

In [1]:
from pathlib import Path
import sys
import numpy as np

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "packages/bellman").is_dir())
sys.path.insert(0, str(ROOT / "packages/bellman"))
from bellman.stage import elaborate_text
application = ROOT / "applications/buffer-stock"
path = application / "declaration/stages/buffer_stock/buffer_stock.bl"

## Elaboration

The stage states the shock laws, consumption choice and backward transition of the bound.

In [2]:
stage = elaborate_text(path.read_text(), str(path))
stage

ElaboratedStage('buffer_stock', symbolic)

## Methodization

The methods file supplies the Euler equation, endogenous grid equations, value derivatives and numerical method choices.

In [3]:
methodized = stage.methodize(path.with_name("methods.bl"))
methodized.dcsn.builder.policy.list()

1  g_cntn_to_dcsn:Aggregation  primitive  → 𝑐≻, 𝑚≻
2  cons_choice:Pullback        primitive  → 𝑐
3  Evaluation                  primitive  → 𝑐

## Calibration

These are the baseline parameter values shown on the example page.

In [4]:
calibration = {"R": 1.03, "β": 0.96, "Γ": 1.03, "ρ": 2.0, "σ_ψ": 0.1, "σ_ξ": 0.1, "p_zero": 0.005}
calibrated = methodized.calibrate(calibration)
calibrated.par

name    set                         value                   describes
β       (0, 1)                      0.96                    discount factor
ρ       (1, inf)                    2.0                     coefficient of relative risk aversion, greater than one
R       ℝ₊₊                         1.03                    gross interest rate
Γ       ℝ₊₊                         1.03                    deterministic permanent income growth factor
σ_ψ     ℝ₊                          0.1                     standard deviation of the log permanent shock
σ_ξ     ℝ₊                          0.1                     standard deviation of the log transitory shock, conditional on a positive draw
p_zero  (0, 1)                      0.005                   probability ℘ of a zero transitory draw
μ_ξ     -log(1 - p_zero) - σ_ξ^2/2  1.2541823544285312e-05  log-mean of the positive transitory draw, at which E[ξ] = 1

## Configuration

In [5]:
configured = calibrated.configure(application / "declaration/settings/buffer_stock.yml")
configured.settings

name        set  value  read at                by              grid
n_a         ℕ    960    policy                 !egm            !Cartesian
extra_grid  ℝ₊   0.001  policy                 !egm            !Cartesian
a_max       Xa   20.0   policy                 !egm            !Cartesian
n_k         ℕ    960    dcsn_to_arvl.evaluate  !Linear         !Cartesian
k_min       Xa   0.001  dcsn_to_arvl.evaluate  !Linear         !Cartesian
k_max       Xa   20.0   dcsn_to_arvl.evaluate  !Linear         !Cartesian
n_ψ_nodes   ℕ    7      dcsn_to_arvl.E_ψ_ξ     !gauss_hermite
n_ξ_nodes   ℕ    7      dcsn_to_arvl.E_ψ_ξ     !gauss_hermite

## Continuation inputs

The illustrative continuation value is a function of saving. The bound is the number zero. The input check retains the conditions it cannot decide; their presence is not a claim that they have been proved.

In [6]:
inputs = {"V[>]": lambda a: -1 / (1 + a), "L": 0.0}
conditions = configured.check_inputs(inputs)
conditions

(Condition(condition='continuation-input', entries=(Identity(kind='field', stage='buffer_stock', address=None, perch='cntn', field='V', owner=None, subjects=(), occurrence=None, mark=None), Named(name='ℝ'), Identity(kind='field', stage='buffer_stock', address=None, perch='cntn', field='a', owner=None, subjects=(), occurrence=None, mark=None)), status='required', evidence=(), explanation='The representing function for V[>] must be measurable on its declared domain, depend only on (a) in that order, and take values in ℝ.'),)

## Default savings grid

The 960 points must equal the former grid from 0.001 to 20.

In [7]:
policy = configured.dcsn.builder.policy.kernels(inputs=inputs)
saving = np.array(policy.grids["policy"].points, dtype=float)[:, 0]
np.testing.assert_allclose(saving, np.linspace(0.001, 20, 960))
print(len(saving), saving[0], saving[-1])

960 0.001 20.0


## Bound transition

The arrival bound is the supremum of Γψ(L−ξ)/R over the shock support. With positive permanent-shock variance it is zero for any supplied L ≤ 0. With zero variance ψ = 1, and the zero-income draw makes it ΓL/R. The terminal bound zero remains zero in both cases.

In [8]:
def evaluate(piece, values):
    return piece.function(*(values[name] for name in piece.arguments), *piece.parameter_values)

for deviation, bound, expected in ((0.1, 0, 0), (0.1, -2, 0), (0, -2, -2 * 1.02 / 1.03)):
    current = elaborate_text(path.read_text(), str(path))
    current = current.methodize(path.with_name("methods.bl"))
    current = current.calibrate({**calibration, "σ_ψ": deviation, "Γ": 1.02})
    current = current.configure(application / "declaration/settings/buffer_stock.yml")
    supplied = {"V[>]": inputs["V[>]"], "L": bound}
    current.check_inputs(supplied)
    record = current.builder["dcsn", "arvl"].kernels(inputs=supplied)
    piece = next(piece for identity, piece in record.output_kernels["Aggregation"].items() if identity.field == "L")
    actual = evaluate(piece, {"L": bound})
    np.testing.assert_allclose(actual, expected)
    print("Permanent-shock deviation:", deviation, "incoming bound:", bound, "arrival bound:", actual)

Permanent-shock deviation: 0.1 incoming bound: 0 arrival bound: 0.0


Permanent-shock deviation: 0.1 incoming bound: -2 arrival bound: 0.0


Permanent-shock deviation: 0 incoming bound: -2 arrival bound: -1.9805825242718447


## Interpolation endpoint

Changing only `extra_grid` to zero makes the first savings point the bound. The power in the Euler block gives consumption zero at infinite marginal value. The named inverse marginal utility retains its finite positive domain. The inverse transition gives resources zero. This point is an endpoint for interpolation, outside the strictly positive consumption choices. For this check the continuation value is 2√max(a, 0); its derivative from positive savings tends to infinity at zero.

In [9]:
endpoint_stage = configured.configure({"extra_grid": 0})
endpoint_inputs = {"V[>]": lambda a: 2*np.sqrt(np.maximum(a, 0.0)), "L": 0.0}
endpoint_conditions = endpoint_stage.check_inputs(endpoint_inputs)
record = endpoint_stage.dcsn.builder.policy.kernels(inputs=endpoint_inputs)
assert record.grids["policy"].points[0] == (0,)
outputs = {identity.field: piece for pieces in record.output_kernels.values() for identity, piece in pieces.items()}
values = {"a": 0.0, "D_{a}(V≻)": np.inf}
consumption = evaluate(outputs["c"], values)
resources = evaluate(outputs["m"], values)
assert consumption == resources == 0
print("Saving:", 0.0, "consumption:", consumption, "resources:", resources)

Saving: 0.0 consumption: 0.0 resources: 0.0


## Observations and decision

Observations:


Decision:


Automated checks and notebook execution do not record the reader’s approval.